<a href="https://colab.research.google.com/github/muskansolanki1122/Brain_Tumor_Detection/blob/main/Brain_Tumor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [21]:
import tensorflow as tf
import numpy as np
import cv2

In [22]:
import zipfile

with zipfile.ZipFile("archive (3).zip", 'r') as zip_ref:
    zip_ref.extractall("data")


In [23]:
dataset_path = "data/Training"

In [24]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2   # 80% train, 20% validation
)

In [25]:
train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

Found 4480 images belonging to 4 classes.


In [26]:
val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 1120 images belonging to 4 classes.


In [27]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [28]:
base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

In [29]:
x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation='relu')(x)
output = Dense(4, activation='softmax')(x)

In [30]:
model = Model(inputs=base.input, outputs=output)



In [31]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [14]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 111s 196ms/step - accuracy: 0.8828 - loss: 0.3281 - val_accuracy: 0.2491 - val_loss: 1.8930
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 21s 147ms/step - accuracy: 0.9563 - loss: 0.1307 - val_accuracy: 0.1545 - val_loss: 2.7224
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 20s 143ms/step - accuracy: 0.9683 - loss: 0.0877 - val_accuracy: 0.2500 - val_loss: 7.6607
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 21s 148ms/step - accuracy: 0.9783 - loss: 0.0686 - val_accuracy: 0.2500 - val_loss: 7.1991
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 22s 155ms/step - accuracy: 0.9821 - loss: 0.0555 - val_accuracy: 0.3304 - val_loss: 1.8493
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 21s 148ms/step - accuracy: 0.9799 - loss: 0.0533 - val_accuracy: 0.2500 - val_loss: 10.0133
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 21s 148ms/step - accuracy: 0.9873 - loss: 0.0425 - val_accuracy: 0.2491 - val_loss: 9.1998
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 21s 152ms/step - accuracy: 0.9904 - loss:

In [15]:
model.save("brain_tumor_model.keras")

In [16]:
from tensorflow.keras.models import Model

feature_extractor = Model(
    inputs=model.input,
    outputs=model.layers[-2].output
)

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(128,)),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')   # risk score
])

ann_model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
import numpy as np
import tensorflow as tf

def get_gradcam(model, img_array):

    last_conv = [layer.name for layer in model.layers if len(layer.output_shape)==4][-1]

    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv, pred = grad_model(img_array)
        class_idx = tf.argmax(pred[0])
        loss = pred[:, class_idx]

    grads = tape.gradient(loss, conv)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))

    conv = conv[0]
    heatmap = conv @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-8)

    return heatmap.numpy()

In [19]:
import cv2
import matplotlib.pyplot as plt

def show_heatmap(img_path, heatmap):

    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))

    heatmap = cv2.resize(heatmap, (224,224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

    plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()